# IterGen before and after the `_is_valid` fix, on the 8 Qwen2.5 tasks it lost

**Runtime: A100 GPU.** Then **Runtime > Run all**. Nothing to edit. About 15 minutes.

Commit ace68e3 encodes the remainder to bytes in IterGen's `_is_valid`, so unfinished strings are
no longer judged invalid. Its parent, 68de350, is the IterGen of the report (the IterGen code did
not change between 2c14eef and 68de350). This runs IterGen (checks off and on) with **both**
commits on the 8 tasks that unconstrained decoding passes and IterGen failed (Qwen2.5-3B-Instruct),
on the same GPU, so before and after are compared on the same hardware. The report's runs were on
G4; the table also shows whether the A100 "before" output equals the G4 one.

Output on Drive: `final/Qwen2.5-3B-Instruct/<mode>_fix8_A100_{before_68de350,after_ace68e3}/`.
The runtime disconnects itself at the end.

In [ ]:
REPO_URL   = 'https://github.com/BowenX307/constrained-llm-js-codegen-final.git'
COMMITS    = {'before': '68de3503ff34f89b3c94b44933bb31b708a9bedb', 'after': 'ace68e307c0eb2002e667128acc7edaf5e6d9161'}
EXPECT_GPU = 'A100'
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
TASKS = ['mbpp_230_replace_blank', 'mbpp_285_text_match_two_three', 'mbpp_604_reverse_words',
         'mbpp_628_replace_spaces', 'mbpp_757_count_reverse_pairs', 'mbpp_771_check_expression',
         'mbpp_772_remove_length', 'mbpp_787_text_match_three']
MODES = ['itergen_off', 'itergen_on']
REPORT_TAG = '2c14eef'          # the G4 runs in the report
AUTO_DISCONNECT = True

In [ ]:
import os, sys, shutil, subprocess, time
CACHE = '/content/cache'
DRIVE_ROOT = '/content/drive/MyDrive/constrained-llm-js-codegen'
MS_DRIVE = f'{DRIVE_ROOT}/mask_store_zst'
os.makedirs(CACHE, exist_ok=True)
smi = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
GPU_NAME = smi.stdout.strip().splitlines()[0] if smi.returncode == 0 and smi.stdout.strip() else ''
print('GPU', GPU_NAME or 'none')
if EXPECT_GPU not in GPU_NAME:
    raise RuntimeError(f'This runtime has {GPU_NAME or "no GPU"}; use {EXPECT_GPU} (Runtime > Change runtime type).')
from google.colab import drive
drive.mount('/content/drive')
CODE = {}                       # 'before' / 'after' -> a checkout of that commit
for label, commit in COMMITS.items():
    path = f'/content/code_{label}'
    if os.path.isdir(path):
        shutil.rmtree(path)
    for cmd in (['git', 'init', '-q', path], ['git', '-C', path, 'remote', 'add', 'origin', REPO_URL],
                ['git', '-C', path, 'fetch', '-q', '--depth', '1', 'origin', commit],
                ['git', '-C', path, 'checkout', '-q', 'FETCH_HEAD']):
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(' '.join(cmd) + ' failed:\n' + r.stderr)
    CODE[label] = path
REPO = CODE['after']
def pip(*a):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *a], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('pip install failed: ' + ' '.join(a) + '\n' + r.stderr[-2000:])
from importlib.metadata import version
if version('transformers') != '5.20.1':
    was = version('transformers')
    pip('transformers==5.20.1')
    if 'transformers' in sys.modules:
        raise RuntimeError(f'transformers was {was}; installed 5.20.1. Runtime > Restart session, then Run all.')
pip('lark', 'regex', 'networkx', 'fire', 'interegular', 'tqdm', 'datasets', 'jsonschema', 'psutil')
if subprocess.run([sys.executable, '-c', 'from syncode.parsers.itergen_parser import IGParser'],
                  cwd=REPO, capture_output=True).returncode != 0:
    pip('--no-deps', '--ignore-requires-python', '--force-reinstall',
        'git+https://github.com/structuredllm/syncode.git@4d6c110a028d3fb9cbb5f1a6ac6446b696f3eac4')
if not shutil.which('zstd'):
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'zstd'], capture_output=True)
ENV = dict(os.environ, SYNCODE_CACHE=CACHE + '/syncode/', HF_HOME=CACHE + '/hf_home', PYTHONPATH=REPO,
           PYTHONUNBUFFERED='1', TOKENIZERS_PARALLELISM='false', HF_HUB_DISABLE_PROGRESS_BARS='1')
print('ready:', {k: v[:7] for k, v in COMMITS.items()})

In [ ]:
def restore_mask_store(model):
    grammar = f'{REPO}/generators/grammars/javascript.lark'
    probe = subprocess.run([sys.executable, '-c',
        'import syncode.common as c; from syncode.parsers.grammars import Grammar\n'
        f'tok = c.load_tokenizer({model!r}); g = Grammar({grammar!r})\n'
        'print("MS_PATH=" + c.SYNCODE_CACHE + "mask_stores/" + type(tok).__name__ + '
        '"/grammar_strict_" + g.hash() + "_" + str(tok.vocab_size) + ".pkl")'],
        env=ENV, cwd=REPO, capture_output=True, text=True)
    lines = [l for l in probe.stdout.splitlines() if l.startswith('MS_PATH=')]
    if probe.returncode != 0 or not lines:
        raise RuntimeError('could not resolve the mask store path:\n' + probe.stderr[-2000:])
    path = lines[-1][len('MS_PATH='):]
    zst = f'{MS_DRIVE}/{os.path.basename(path)}.zst'
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if not (os.path.exists(path) and os.path.getsize(path) > 100 * 2**20):
        if not os.path.exists(zst):
            raise RuntimeError(f'mask store not on Drive: {zst}')
        r = subprocess.run(['zstd', '-q', '-d', '-f', '-T0', zst, '-o', path + '.part'], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError('the copy on Drive is unreadable: ' + r.stderr[-300:])
        os.replace(path + '.part', path)
    print('mask store ready:', os.path.basename(path))

import json
def load_run(run_dir):
    p = f'{run_dir}/eval.jsonl'
    status = {json.loads(l)['name']: json.loads(l)['status'] for l in open(p)} if os.path.exists(p) else {}
    comp = {n: json.load(open(f'{run_dir}/{n}.json'))['completion'] for n in TASKS if os.path.exists(f'{run_dir}/{n}.json')}
    return status, comp

slug = MODEL.split('/')[-1]
restore_mask_store(MODEL)       # the grammar is the same in both commits, so is the mask store
tasks_file = '/content/work/tasks.txt'
os.makedirs('/content/work', exist_ok=True)
open(tasks_file, 'w').write('\n'.join(TASKS) + '\n')
quiet = ('Warning', 'warn(', 'Loading weights', 'torch_dtype', 'httpx')
try:
    unc, _ = load_run(f'{DRIVE_ROOT}/final/{slug}/unconstrained_{REPORT_TAG}')
    for mode in MODES:
        res = {}
        for label, path in CODE.items():
            name = f'{mode}_fix8_A100_{label}_{COMMITS[label][:7]}'
            out = f'/content/work/{slug}/{name}'
            r = subprocess.run([sys.executable, 'run.py', '--model', MODEL, '--mode', mode, '--out', out,
                                '--tasks_file', tasks_file], cwd=path, env=dict(ENV, PYTHONPATH=path),
                               text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
            print('\n'.join(l for l in r.stdout.splitlines() if not any(s in l for s in quiet))[-2000:])
            if r.returncode != 0:
                raise RuntimeError(f'{mode} {label}: run.py failed')
            if shutil.which('node'):
                subprocess.run([sys.executable, 'evaluate.py', out], cwd=path, capture_output=True, text=True)
            shutil.copytree(out, f'{DRIVE_ROOT}/final/{slug}/{name}', dirs_exist_ok=True)
            res[label] = load_run(out)
        g4_status, g4_comp = load_run(f'{DRIVE_ROOT}/final/{slug}/{mode}_{REPORT_TAG}')
        print(f'\n== {mode}  (saved to Drive final/{slug}/{mode}_fix8_A100_*)')
        print(f'{"task":32} {"unconstr. G4":14} {"report G4":16} {"before A100":16} {"after A100":16} before=G4?')
        for n in TASKS:
            b, a = res['before'], res['after']
            print(f'{n:32} {unc.get(n, "?"):14} {g4_status.get(n, "?"):16} {b[0].get(n, "(no node)"):16} '
                  f'{a[0].get(n, "(no node)"):16} {b[1].get(n) == g4_comp.get(n)}')
        print()
        for n in TASKS:
            d = json.load(open(f'/content/work/{slug}/{mode}_fix8_A100_after_{COMMITS["after"][:7]}/{n}.json'))
            print(f'--- {n}, after the fix  (mask applied {d["stats"].get("mask_applied")} times)')
            print(d['completion'].rstrip())
    print('ALL DONE')
finally:
    if AUTO_DISCONNECT:
        from google.colab import drive, runtime
        drive.flush_and_unmount()
        time.sleep(30)
        runtime.unassign()